In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
n = 1000

jarak_km = np.random.exponential(scale=15, size=n).clip(0.5, 150)

# --- 1b. Status kedekatan ---
status_kedekatan = np.random.choice(
    ['Keluarga', 'Teman Sekolah', 'Teman Kerja', 'Tetangga'],
    size=n,
    p=[0.30, 0.25, 0.25, 0.20]
)

# --- 1c. Rentang usia ---
rentang_usia = np.random.choice(
    ['18-25', '26-35', '36-50', '>50'],
    size=n,
    p=[0.20, 0.35, 0.30, 0.15]
)

# --- 1d. Membawa pasangan ---
membawa_pasangan = np.random.binomial(n=1, p=0.55, size=n)

# --- 1e. Waktu respon RSVP (hari) ---
# Eksponensial: mayoritas respon cepat (1-5 hari), sedikit yang lambat
waktu_respon_hari = np.random.exponential(scale=4, size=n).clip(0.5, 30)

# ============================================================
# 2. GENERATE TARGET (y) MENGGUNAKAN LOGIT MODEL
# ============================================================

# --- 2a. Encoding variabel kategorikal untuk logit ---
is_keluarga      = (status_kedekatan == 'Keluarga').astype(int)
is_teman_sekolah = (status_kedekatan == 'Teman Sekolah').astype(int)
is_tetangga      = (status_kedekatan == 'Tetangga').astype(int)
# Referensi: Teman Kerja

is_18_25 = (rentang_usia == '18-25').astype(int)
is_26_35 = (rentang_usia == '26-35').astype(int)
is_36_50 = (rentang_usia == '36-50').astype(int)
# Referensi: >50

# --- 2b. Standard scaling untuk fitur kontinu ---
jarak_km_mean, jarak_km_std = jarak_km.mean(), jarak_km.std()
waktu_respon_mean, waktu_respon_std = waktu_respon_hari.mean(), waktu_respon_hari.std()

jarak_km_scaled = (jarak_km - jarak_km_mean) / jarak_km_std
waktu_respon_scaled = (waktu_respon_hari - waktu_respon_mean) / waktu_respon_std

# --- 2c. Logit linear ---
# Koefisien dipilih agar menghasilkan ~70% hadir, ~30% tidak hadir
logit = (
    0.0 +                     # intercept
    -0.6 * jarak_km_scaled +  # semakin jauh, semakin kecil probabilitas hadir
    +0.8 * is_keluarga +      # keluarga paling mungkin hadir
    +0.3 * is_teman_sekolah +
    -0.4 * is_tetangga +      # tetangga paling mungkin tidak hadir
    -0.3 * is_18_25 +         # usia muda, lebih fluktuatif
    +0.7 * is_26_35 +         # usia produktif, paling mungkin hadir
    +0.4 * is_36_50 +
    +0.5 * membawa_pasangan + # membawa pasangan meningkatkan komitmen
    -0.5 * waktu_respon_scaled  # respon cepat (negatif) = lebih mungkin hadir
)

# --- 2d. Sigmoid -> probabilitas -> sampling Bernoulli ---
prob = 1 / (1 + np.exp(-logit))
prob = np.clip(prob, 0.01, 0.99)
hadir = np.random.binomial(n=1, p=prob)

# ============================================================
# 3. GABUNGKAN KE DATAFRAME
# ============================================================
df = pd.DataFrame({
    'jarak_km':              jarak_km.round(1),
    'status_kedekatan':      status_kedekatan,
    'rentang_usia':          rentang_usia,
    'membawa_pasangan':      membawa_pasangan,
    'waktu_respon_hari':     waktu_respon_hari.round(1),
    'hadir':                 hadir
})

# ============================================================
# 4. EKSPLORASI & VALIDASI
# ============================================================
print("=== 5 BARIS PERTAMA ===")
print(df.head())
print()

print("=== STATISTIK DESKRIPTIF ===")
print(df.describe())
print()

print("=== DISTRIBUSI TARGET ===")
print(df['hadir'].value_counts())
print(f"Proporsi hadir: {df['hadir'].mean():.1%}")
print(f"Proporsi tidak hadir: {1 - df['hadir'].mean():.1%}")
print()

print("=== DISTRIBUSI FITUR KATEGORIKAL vs TARGET ===")
for col in ['status_kedekatan', 'rentang_usia', 'membawa_pasangan']:
    print(f"\n--- {col} ---")
    print(df.groupby(col)['hadir'].agg(['count', 'mean']).round(3))

=== 5 BARIS PERTAMA ===
   jarak_km status_kedekatan rentang_usia  membawa_pasangan  \
0       7.0         Keluarga        26-35                 0   
1      45.2    Teman Sekolah        26-35                 0   
2      19.8         Tetangga          >50                 1   
3      13.7      Teman Kerja        26-35                 0   
4       2.5         Tetangga        26-35                 0   

   waktu_respon_hari  hadir  
0                3.4      1  
1                6.5      0  
2                5.7      1  
3                0.7      1  
4                0.6      0  

=== STATISTIK DESKRIPTIF ===
         jarak_km  membawa_pasangan  waktu_respon_hari        hadir
count  1000.00000       1000.000000        1000.000000  1000.000000
mean     14.59670          0.555000           3.935000     0.653000
std      14.58132          0.497214           3.867714     0.476254
min       0.50000          0.000000           0.500000     0.000000
25%       4.07500          0.000000           1

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv('rsvp_wedding_data.csv')

print("=== 5 Baris Pertama Data ===")
df.head()

X = df.drop('hadir', axis=1)
y = df['hadir']

X_encoded = pd.get_dummies(X, columns=['status_kedekatan'], drop_first=True, dtype=int)

print("\n=== Data Setelah One-Hot Encoding ===")
print(X_encoded.head())

X_train, X_test, y_train, y_test = train_test_split(
   X_encoded, y, test_size=0.2, random_state=42
)

kolom_numerik = ['jarak_domisili_km', 'usia', 'waktu_respons_hari']


scaler = StandardScaler()

X_train[kolom_numerik] = scaler.fit_transform(X_train[kolom_numerik])

X_test[kolom_numerik] = scaler.transform(X_test[kolom_numerik])


model = LogisticRegression(class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\n=== Hasil Evaluasi Model ===")
print(f"Akurasi: {accuracy_score(y_test, y_pred):.2f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Tidak Hadir', 'Hadir']))


=== 5 Baris Pertama Data ===

=== Data Setelah One-Hot Encoding ===
   jarak_domisili_km  usia  bawa_pasangan  waktu_respons_hari  \
0                 33    31              1                   3   
1                 17    27              1                   6   
2                 38    38              0                   1   
3                 92    48              1                   1   
4                 15    18              1                   1   

   status_kedekatan_Lainnya  status_kedekatan_Teman Kerja  \
0                         0                             0   
1                         0                             0   
2                         0                             1   
3                         0                             1   
4                         0                             0   

   status_kedekatan_Teman Sekolah  
0                               0  
1                               0  
2                               0  
3                             

Feature Engineering

In [31]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv('rsvp_wedding_data.csv')

df = df.drop('usia', axis=1)

df['kategori_jarak'] = pd.cut(
    df['jarak_domisili_km'], 
    bins=[0, 20, 100, 1000], 
    labels=['Dekat', 'Menengah', 'Jauh']
)

df['kategori_respons'] = pd.cut(
    df['waktu_respons_hari'], 
    bins=[0, 3, 14, 30], 
    labels=['Cepat', 'Sedang', 'Lambat']
)

df = df.drop(['jarak_domisili_km', 'waktu_respons_hari'], axis=1)

X = df.drop('hadir', axis=1)
y = df['hadir']

X_encoded = pd.get_dummies(X, drop_first=True, dtype=int)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

model = LogisticRegression(class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\n=== Hasil Evaluasi Model ===")
print(f"Akurasi: {accuracy_score(y_test, y_pred):.2f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Tidak Hadir', 'Hadir']))


=== Hasil Evaluasi Model ===
Akurasi: 0.74

Classification Report:
              precision    recall  f1-score   support

 Tidak Hadir       0.50      0.65      0.56        51
       Hadir       0.87      0.78      0.82       149

    accuracy                           0.74       200
   macro avg       0.68      0.71      0.69       200
weighted avg       0.77      0.74      0.75       200



Feature Engineering + Hyperparameter tuning

In [32]:
from sklearn.model_selection import GridSearchCV

base_model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=2000)

param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'l1_ratio': [0, 1],        # 0 = L2, 1 = L1
    'solver': ['liblinear', 'saga']
}

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n=== Laporan Optimasi Model ===")
print(f"Parameter Terbaik : {grid_search.best_params_}")

print("\n=== Hasil Evaluasi Model ===")
print(f"Akurasi: {accuracy_score(y_test, y_pred):.2f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Tidak Hadir', 'Hadir']))


=== Laporan Optimasi Model ===
Parameter Terbaik : {'C': 0.1, 'l1_ratio': 0, 'solver': 'saga'}

=== Hasil Evaluasi Model ===
Akurasi: 0.76

Classification Report:
              precision    recall  f1-score   support

 Tidak Hadir       0.52      0.65      0.58        51
       Hadir       0.87      0.80      0.83       149

    accuracy                           0.76       200
   macro avg       0.70      0.72      0.71       200
weighted avg       0.78      0.76      0.77       200



Undersampling

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import resample

df = pd.read_csv('rsvp_wedding_data.csv')
df = df.drop('usia', axis=1)

df['kategori_jarak'] = pd.cut(
    df['jarak_domisili_km'], 
    bins=[0, 20, 100, 1000], 
    labels=['Dekat', 'Menengah', 'Jauh']
)

df['kategori_respons'] = pd.cut(
    df['waktu_respons_hari'], 
    bins=[0, 3, 14, 30], 
    labels=['Cepat', 'Sedang', 'Lambat']
)

df = df.drop(['jarak_domisili_km', 'waktu_respons_hari'], axis=1)

df_hadir = df[df['hadir'] == 1]
df_tidak_hadir = df[df['hadir'] == 0]

print(f"Jumlah sampel sebelum balancing:")
print(f"  Hadir       : {len(df_hadir)}")
print(f"  Tidak Hadir : {len(df_tidak_hadir)}")

if len(df_hadir) > len(df_tidak_hadir):
    df_hadir_downsampled = resample(
        df_hadir,
        replace=False,
        n_samples=len(df_tidak_hadir),
        random_state=42
    )
    df_balanced = pd.concat([df_hadir_downsampled, df_tidak_hadir])
    print(f"\nHadir diturunkan dari {len(df_hadir)} menjadi {len(df_hadir_downsampled)}")
else:
    df_tidak_hadir_downsampled = resample(
        df_tidak_hadir,
        replace=False,
        n_samples=len(df_hadir),
        random_state=42
    )
    df_balanced = pd.concat([df_hadir, df_tidak_hadir_downsampled])
    print(f"\nTidak Hadir diturunkan dari {len(df_tidak_hadir)} menjadi {len(df_tidak_hadir_downsampled)}")

print(f"Total sampel setelah balancing: {len(df_balanced)}")
print(f"  Hadir       : {len(df_balanced[df_balanced['hadir'] == 1])}")
print(f"  Tidak Hadir : {len(df_balanced[df_balanced['hadir'] == 0])}")

# Lanjutkan dengan data yang sudah balanced
X = df_balanced.drop('hadir', axis=1)
y = df_balanced['hadir']

X_encoded = pd.get_dummies(X, drop_first=True, dtype=int)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

base_model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=2000)

param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'l1_ratio': [0, 1],       
    'solver': ['liblinear', 'saga']
}

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n=== Laporan Optimasi Model ===")
print(f"Parameter Terbaik : {grid_search.best_params_}")

print("\n=== Hasil Evaluasi Model ===")
print(f"Akurasi: {accuracy_score(y_test, y_pred):.2f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Tidak Hadir', 'Hadir']))

Jumlah sampel sebelum balancing:
  Hadir       : 748
  Tidak Hadir : 252

Hadir diturunkan dari 748 menjadi 252
Total sampel setelah balancing: 504
  Hadir       : 252
  Tidak Hadir : 252

=== Laporan Optimasi Model ===
Parameter Terbaik : {'C': 1, 'l1_ratio': 0, 'solver': 'liblinear'}

=== Hasil Evaluasi Model ===
Akurasi: 0.78

Classification Report:
              precision    recall  f1-score   support

 Tidak Hadir       0.83      0.74      0.78        54
       Hadir       0.74      0.83      0.78        47

    accuracy                           0.78       101
   macro avg       0.78      0.79      0.78       101
weighted avg       0.79      0.78      0.78       101

